## Train / Validation / Test Splits
Today in this notebook I will learn how to divide our data into three distinct groups:
- Training set
- Validation set
- test set
the training set is used to train the model while the validation set helps us tune the model and the test set is used just once ,all this at the end of the process to test the accuracy of the model. 

## Why to split the test set?
The primary reson behind separating the test set from the rest is that we should not use it during the model creation phase.

## Import Libraries


In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

## Load the dataset
The student performance dataset will be used for this day exercises the aim is to predict the students final exam score.
This ensures that we can concentrate on the topic of validation sets rather than having to learn a whole new dataset.

In [4]:
df=pd.read_csv("student_performance_dataset.csv")


## Explor the dataset
This dataset contains the student attributes which will be using to predict the final exam score and befor we split our data into train-test we need to enusre that there is no missing value issue.

In [5]:
df.head(10)

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D
5,6,Female,4.2,78.2,5.7,High School,Yes,No,No,79.8,81.6,B
6,7,Male,1.5,100.0,5.0,High School,Yes,Yes,Yes,58.6,69.6,D
7,8,Male,6.2,86.4,6.0,High School,Yes,Yes,No,78.4,88.0,B
8,9,Male,5.3,81.3,6.7,Bachelors,Yes,No,No,63.8,84.4,B
9,10,Female,2.8,86.8,5.1,Bachelors,Yes,Yes,No,57.7,74.3,C


In [6]:
df.shape

(1000, 12)

In [7]:
df.isnull().sum()

student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                0
final_grade                     0
dtype: int64

In [8]:
df["parental_education"]=df["parental_education"].fillna(df["parental_education"].mode()[0])

### Data Cleaning Note
The dataset contains 102 missing values in the **parental_education column** These were handled by filling the missing values with the mode (the most frequent value) to ensure data integrity for further analysis.

## Define Features and Target 
our target variable is **Final_exam_score** this is the value the model will try to predict('y'), and all other attribute will be used as input features ('x')

In [13]:
x = df.drop(columns=['final_exam_score', 'final_grade'])
y = df['final_exam_score']
print("x shape is",x.shape)
print("y shape is",y.shape)

x shape is (1000, 10)
y shape is (1000,)


In [14]:
x.dtypes

student_id                      int64
gender                            str
study_time_hours              float64
attendance_percent            float64
sleep_hours                   float64
parental_education                str
internet_access                   str
extracurricular_activities        str
part_time_job                     str
previous_grade                float64
dtype: object

In [15]:
x=pd.get_dummies(x,drop_first=True)

**Final_exam_score**,**Final_grage** are not included in the list of features since they are dependent on the target variable ,including these two variables as part of the features will enable the machine learning model to have knowledge about the solution leading to data leakage.
Categorical columns were converted into numerical so machine learning model can process them.

## create the three-way split
In this case rather than doing the train-test split we will do the following three splits:
- 60% Training
- 20% Validation
- 20% Testing
This will require us to make two calls to 'train_test_split'. 
first separate 20% of the intial dataset as the test dataset.
afterwards we split the rest 80% into the training and validation sets.
the validation set will be 25% of the rest dataset.

In [17]:
x_temp,x_test,y_temp,y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [18]:
x_train,x_val,y_train,y_val=train_test_split(x_temp,y_temp,test_size=0.25,random_state=42)

In [20]:
print("Training set:",x_train.shape)
print("Validation set:",x_val.shape)
print("Test set:",x_test.shape)

Training set: (600, 12)
Validation set: (200, 12)
Test set: (200, 12)


We must verify that our splits has the correct sizes.
since our split is done with a **random_state** set to a fixed value,it will choose the same rows every time the notebook runs.

In [21]:
model=DecisionTreeRegressor(max_depth=3,random_state=42)
model.fit(x_train,y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max

## Evaluate the model on the validation set
In this phase we should not perform any evaluation with the test set
we are in the process of building and refining the model so we can use the validation set for that purpose
Here we will compute the R^2 value which indicated the goodness of fit of the model.

In [22]:
y_value_predict=model.predict(x_val)
value_r2=r2_score(y_val,y_value_predict)
print("Validation R^2 is:",value_r2)

Validation R^2 is: 0.28459088593517945


In [25]:
value_mae=mean_absolute_error(y_val,y_value_predict)
value_rmse=np.sqrt(mean_squared_error(y_val,y_value_predict))
value_r2=r2_score(y_val,y_value_predict)
print("Validation MAE=",value_mae)
print("Validation RMSE=",value_rmse)
print("Validation r^2=",value_r2)


Validation MAE= 7.126390944874386
Validation RMSE= 9.184502337837975
Validation r^2= 0.28459088593517945


### Model Validation Metrics
After validation of the model using the validation dataset, the performance metrics obtained include the following:
MAE (Mean Absolute Error): 7.12
This metric measures the average absolute error in the predictions made by the model.
RMSE (Root Mean Squared Error): 9.18
This metric gives an idea about the magnitude of error in the predictions and penalizes large errors.
R^2 Score:0.284 
This score indicates that the model can account for 28.4% of the variance in the target variable.
R^2 Score:0.284 
For tuning we will focuse on the validation R^2 score and for R^2 a higher value is better.

## Tune one Hyperparameter
we will tune one setting of the Decision Tree: max_depth,insted of choosing the best depth based on the test set we will train several models and compare there performance.
we will test:
- max_depth=2
- max_depth=3
- max_depth=5
- max_depth=7
- max_depth=10
the test set will remain the same.

In [29]:
depth=[2,3,5,7,10]
result=[]
for d in depth:
    model=DecisionTreeRegressor(max_depth=d,random_state=42)
    model.fit(x_train,y_train)
    y_value_predict=model.predict(x_val)
    value_mae=mean_absolute_error(y_val,y_value_predict)
    value_rmse=np.sqrt(mean_squared_error(y_val,y_value_predict))
    value_r2=r2_score(y_val,y_value_predict)
    result.append({
    "max_depth":d,
    "Validation MAE=":value_mae,
    "Validation RMSE=":value_rmse,
    "Validation R^2=":value_r2
    })
    

In [30]:
result_df=pd.DataFrame(result)
result_df

,max_depth,Validation MAE=,Validation RMSE=,Validation R^2=
0,2,7.065339,9.116246,0.295185
1,3,7.126391,9.184502,0.284591
2,5,6.968444,9.112262,0.295801
3,7,7.190807,9.423781,0.246829
4,10,7.589804,9.760256,0.192085


Optimal performance: The best results were received with the max_depth of 5, which corresponded to the smallest Validation MAE (6.968), smallest - - Validation RMSE (9.112), and maximum R² (0.296).
- Underfitting: With smaller max_depth values (2 and 3), the model is somewhat simpler, and, therefore, the error is larger.
- Overfitting: With increased depth (max_depth of 7 and 10), the errors become larger (MAE and RMSE), and the R² score is much lower (only 0.192 with the max_depth of 10).
Conclusion: max_depth = 5 is chosen as the optimal hyperparameter for the final model.
This decision was made using the validation set only ,we have still not used the test set.

## Train the Final Model
with the optimal hyperparameter obtained from the validation set we are ready to define the final model ,the curical issue here is that the model structure is now fixed,and no further decisions will be made based on the test set.

In [31]:
final_model=DecisionTreeRegressor(max_depth=5,random_state=42)
final_model.fit(x_train,y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max

## Final Evaluation on Test set
the model parameters have now been selected ,it is now the time to evaluate the model using the test set.
The test set will be used for this purpose for the first time and only once,the test score gives a more realistic idea of the performance of the final model.

In [ ]:
y_test_predict=final_model.predict(x_test)
test_mae=mean_absolute_error(y_test,y_test_predict)
test_rmse=np.sqrt(mean_squared_error(y_test,y_test_predict))
test_r2=r2_score(y_test,y_test_predict)
print("test MAE=",test_mae)
print("test RMSE=",test_rmse)
print("test r^2=",test_r2)


test MAE= 6.285587754500044
test RMSE= 7.9731364604251835
test r^2= 0.4093951896688438


## Performance of the Final Model
The final model was chosen based on the validation performance and tested only using the unseen test dataset.
The validation score was used for choosing the model configuration while the test score was used for evaluating the model only at the end,thses two scores do not have to match because they were calculated using different subsets.

In [33]:
final_result=pd.DataFrame({"Metric":["MAE","RMSE","R2"],
                           "Test_Score":[test_mae,test_rmse,test_r2]
                           })
final_result

,Metric,Test_Score
0,MAE,6.285588
1,RMSE,7.973136
2,R2,0.409395


## Compare Validation VS Test
The validation score is employed when selecting the model setup ,wherease the test score is used only for evaluation.
It is natural that the scores are different because the validation set and the test set consist of different samples.

In [34]:
final_results = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Validation_Score": [value_mae, value_rmse, value_r2],
    "Test_Score": [test_mae, test_rmse, test_r2]
})

final_results

,Metric,Validation_Score,Test_Score
0,MAE,7.589804,6.285588
1,RMSE,9.760256,7.973136
2,R2,0.192085,0.409395


## What would go wrong if you had tuned against the test set instead?
If we evaluated various models or hyperparameters against the test set and then picked the one with the best test score then we should use information from the test set in order to choose our model,the process of model selection would be optimized with respect to the particular test set used then the test set would no longer be completely unseen data.
The test score would be optimistic and would not properly reflect the performance of the model in real world.
The right way of doing things is the following:
- Training set--> learning the model.
- Validation set--> Tuning and making decisions.
- Test set--> Testing only.